# 01 — KAIST Acoustic Dataset Inventory (EDA-only)

**Objective:** Inspect and inventory the downloaded KAIST acoustic `.mat` recordings.  
**Scope:** Read-only dataset understanding/validation. No model training, no production pipeline changes.

- Dataset root (configured): `G:\My Drive\IndustrialSense\datasets\kaist`
- Acoustic directory: `G:\My Drive\IndustrialSense\datasets\kaist\acoustic`
- Uses `KAIST_DATA_ROOT` when set (expects `.../kaist` or `.../kaist/acoustic`).


In [ ]:
import os, json, sys
from pathlib import Path
import pandas as pd
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)

# EDA utils (separate from production backend modules)
EDA_DIR = Path.cwd()
if not (EDA_DIR / 'kaist_acoustic_eda_utils.py').exists():
    # notebook may be opened from repo root
    EDA_DIR = Path.cwd() / 'backend' / 'eda'
sys.path.insert(0, str(EDA_DIR))
from kaist_acoustic_eda_utils import get_acoustic_dir, build_inventory, save_inventory, inspect_mat_structure

os.environ.setdefault('KAIST_DATA_ROOT', r'G:\My Drive\IndustrialSense\datasets\kaist')
ac_dir = get_acoustic_dir()
print('Acoustic directory:', ac_dir)
print('Exists:', ac_dir.exists())


## 1. MATLAB file structure inspection (all `.mat` recordings)

Phase 1 findings are derived recursively from actual files (no guessed channel names/rates/units).
- Top-level key, `Signal` (mat_struct) fields: `x_values`, `y_values`, `function_record`
- `x_values`: `start_value`, `increment`, `number_of_values`, `quantity` (label/unit) — represents the axis (verified as time in seconds when label is `s`)
- `y_values`: `values` (actual signal samples), `quantity` (verified unit/label when present)
- `function_record`: metadata (type, times, channel, annotations, etc.)


In [ ]:
from pathlib import Path
mat_files = sorted([p for p in ac_dir.rglob('*.mat') if p.is_file()]) if ac_dir.exists() else []
print('MAT files found:', len(mat_files))
for p in mat_files:
    info = inspect_mat_structure(p)
    print('\n---', p.name, '---')
    print(json.dumps({k:v for k,v in info.items() if k!='path'}, indent=2, default=str))


## 2. Reproducible dataset inventory

Includes: filename, relative path, file size, fault category, operating condition/load, MATLAB structure, signal field/shape/dtype, verified sampling frequency/duration/units (when available), inspection status, warnings. Missing/corrupted/inconsistent recordings are flagged in status/warnings (not silently discarded).


In [ ]:
df = build_inventory(ac_dir)
out_dir = EDA_DIR / 'outputs'
csv_path, json_path = save_inventory(df, out_dir)
print('Saved:', csv_path)
print('Saved:', json_path)
df[['filename','relative_path','file_size_bytes','fault_category','operating_condition_load','severity_or_variant',
    'matlab_structure','signal_field','signal_shape','data_type','sampling_frequency_hz_verified','duration_s_verified',
    'units_verified','x_unit_verified','x_represents','inspection_status','warnings']]


In [ ]:
# Summary
summary = {
    'n_files': int(len(df)),
    'statuses': df['inspection_status'].value_counts(dropna=False).to_dict(),
    'formats': df['format'].value_counts(dropna=False).to_dict() if 'format' in df else {},
    'fs_values_hz': sorted([float(x) for x in df['sampling_frequency_hz_verified'].dropna().unique().tolist()]) if 'sampling_frequency_hz_verified' in df else [],
    'y_units_verified': sorted([str(x) for x in df['units_verified'].dropna().unique().tolist()]) if 'units_verified' in df else [],
    'x_units_verified': sorted([str(x) for x in df['x_unit_verified'].dropna().unique().tolist()]) if 'x_unit_verified' in df else [],
    'durations_s': sorted([float(x) for x in df['duration_s_verified'].dropna().unique().tolist()]) if 'duration_s_verified' in df else [],
    'fault_categories': df['fault_category'].value_counts(dropna=False).to_dict(),
    'operating_conditions': df['operating_condition_load'].value_counts(dropna=False).to_dict(),
}
print(json.dumps(summary, indent=2))
with open(out_dir/'acoustic_inventory_summary.json','w',encoding='utf-8') as f:
    json.dump(summary,f,indent=2,ensure_ascii=False)
summary


## 3. Notes
- **Signal samples:** `Signal.y_values.values` (verified per-file).  
- **Axis:** `Signal.x_values` with `increment` and `quantity.label='s'` indicates uniform time axis in seconds; sampling frequency derived as `1/increment` when verified.  
- **Metadata:** `Signal.function_record` (and nested fields) provides acquisition/annotation metadata; some fields are char arrays and are normalized in inventory extraction.  
- **Units:** Use only verified labels (e.g. `Pa`, `s`) from file metadata; anything not established is marked unknown/unverified.
